# PHQ-8 Depression Detection — Mid Fusion + Temporal Alignment (Approach A): Mental-RoBERTa + WavLM

**Temporal Approach A** extends mid-fusion by dividing each interview into `N_BUCKETS` equal temporal windows
before extracting embeddings, then applying learned attention pooling over the resulting sequence.

```
Interview timeline (normalised 0 → 1), N_BUCKETS = 5:

  text chunks  ──► split by relative index into 5 windows
                   each window ──► [frozen text LSTM+attn] ──► 64-dim  ──┐
  audio segs   ──► split by relative index into 5 windows               ├──► concat (192-dim)
                   each window ──► [frozen audio LSTM+attn] ──► 128-dim ──┘

  Result: sequence of 5 fused vectors, shape (5, 192)
       ──► learned attention pooling ──► 192-dim
       ──► trainable MLP ──► logit
```

**Frozen encoders**: same pre-trained weights as the flat mid-fusion notebook.
  `mental_roberta_lstm_cls_best.pth` and `wavLM_MLP_LSTM_cls_best.pth`.

**Key difference from flat mid-fusion**: instead of running each encoder over *all* chunks/segments
at once, they are run separately per temporal bucket. This lets the attention layer learn
*which part of the interview* is most diagnostic, rather than collapsing the whole session immediately.

**Assumption**: feature index ≈ temporal position (valid because both modalities sweep
the interview linearly with a sliding window).

**Trainable parameters**: attention weights (192) + joint MLP (~33 K) only.

## 1. Install Dependencies

In [ ]:
!pip install transformers tqdm pandas numpy scikit-learn matplotlib --quiet
print("All packages ready.")

## 2. Imports & Configuration

In [ ]:
import os
import random

import numpy as np
import pandas as pd
from pathlib import Path
from collections import defaultdict

import torch
import torch.nn as nn
from torch.optim import Adam

import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report
)
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

print("Imports successful.")

In [ ]:
# --- Reproducibility ---
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

# --- Paths ---
BASE_DIR      = Path("..")
DATASET_DIR   = BASE_DIR / "dataset"
PROCESSED_DIR = BASE_DIR / "processed"

TRAIN_CSV = DATASET_DIR / "train_split_Depression_AVEC2017.csv"
DEV_CSV   = DATASET_DIR / "dev_split_Depression_AVEC2017.csv"
TEST_CSV  = DATASET_DIR / "full_test_split.csv"

TEXT_FEATURE_CACHE     = PROCESSED_DIR / "mental_roberta_features_cls.npz"
EDA_TEXT_FEATURE_CACHE = PROCESSED_DIR / "mental_roberta_features_eda_cls.npz"
AUDIO_FEATURE_CACHE    = PROCESSED_DIR / "wavlm_features_cls.npz"

TEXT_CKPT  = BASE_DIR / "experiments" / "best_model" / "mental_roberta_lstm_cls" / "mental_roberta_lstm_cls_best.pth"
AUDIO_CKPT = BASE_DIR / "experiments" / "best_model" / "wavlm_mlp_lstm_cls" / "wavLM_MLP_LSTM_cls_best.pth"

SAVE_DIR = BASE_DIR / "experiments" / "best_model" / "mid_fusion_temporal_a_mentalroberta_wavlm"
SAVE_DIR.mkdir(parents=True, exist_ok=True)

# --- Temporal alignment ---
# Number of equal-length temporal windows to divide each interview into.
# Each window produces one (text_emb, audio_emb) pair from the frozen encoders.
# Kept small (5) so each bucket still has enough chunks/segments for the LSTM.
#   ~19 text chunks  / 5 = ~4 chunks per bucket
#   ~106 audio segs  / 5 = ~21 segments per bucket
N_BUCKETS = 5

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Seed: {SEED}  |  Device: {DEVICE}  |  N_BUCKETS: {N_BUCKETS}")
print(f"Text ckpt  : {TEXT_CKPT.resolve()}")
print(f"Audio ckpt : {AUDIO_CKPT.resolve()}")

## 3. Load Label Files

In [ ]:
train_df = pd.read_csv(TRAIN_CSV)[["Participant_ID", "PHQ8_Binary"]]
dev_df   = pd.read_csv(DEV_CSV)[["Participant_ID", "PHQ8_Binary"]]
test_df  = pd.read_csv(TEST_CSV)[["Participant_ID", "PHQ_Binary"]].rename(
    columns={"PHQ_Binary": "PHQ8_Binary"}
)

train_labels = dict(zip(train_df.Participant_ID, train_df.PHQ8_Binary))
dev_labels   = dict(zip(dev_df.Participant_ID,   dev_df.PHQ8_Binary))
test_labels  = dict(zip(test_df.Participant_ID,  test_df.PHQ8_Binary))
pid_to_split = (
    {pid: "train" for pid in train_labels}
    | {pid: "dev"   for pid in dev_labels}
    | {pid: "test"  for pid in test_labels}
)

for name, labels in [("Train", train_labels), ("Dev", dev_labels), ("Test", test_labels)]:
    dep = sum(labels.values())
    print(f"{name:5s}: {len(labels):3d} participants  depressed: {dep}  control: {len(labels)-dep}")

## 4. Load Pre-extracted Features

In [ ]:
def load_participant_features(cache_path: Path, label: str, eda_cache_path: Path = None):
    assert cache_path.exists(), f"Cache not found: {cache_path}"
    print(f"Loading {label} from {cache_path.name} ...")
    cache = np.load(cache_path, allow_pickle=False)
    in_memory = {
        "train": defaultdict(lambda: {"feats": [], "label": None}),
        "dev":   defaultdict(lambda: {"feats": [], "label": None}),
        "test":  defaultdict(lambda: {"feats": [], "label": None}),
    }
    for feat, lbl, pid, split_b in zip(
        cache["feats"], cache["binary_labels"], cache["pids"], cache["splits"]
    ):
        s = split_b.decode(); pid = int(pid)
        in_memory[s][pid]["feats"].append(feat)
        in_memory[s][pid]["label"] = int(lbl)

    if eda_cache_path is not None and eda_cache_path.exists():
        print(f"  + EDA features from {eda_cache_path.name}")
        eda = np.load(eda_cache_path, allow_pickle=False)
        eda_virtual = defaultdict(lambda: {"feats": [], "label": None})
        for feat, lbl, pid, aug_i in zip(
            eda["feats"], eda["binary_labels"], eda["pids"], eda["aug_indices"]
        ):
            vpid = int(pid) * 1000 + int(aug_i)
            eda_virtual[vpid]["feats"].append(feat)
            eda_virtual[vpid]["label"] = int(lbl)
        for vpid, data in eda_virtual.items():
            in_memory["train"][vpid] = data
        n_dep = sum(d["label"] for d in eda_virtual.values())
        print(f"  Added {len(eda_virtual)} EDA virtual participants "
              f"(dep: {n_dep}  ctrl: {len(eda_virtual)-n_dep})")

    splits_list = {s: list(in_memory[s].items()) for s in ("train", "dev", "test")}
    for s, name in [("train", "train"), ("dev", "val"), ("test", "test")]:
        n_p = len(splits_list[s])
        n_d = sum(d["label"] for _, d in splits_list[s])
        print(f"  {name:5s}: {n_p} participants  dep: {n_d}  ctrl: {n_p-n_d}")
    return splits_list


text_splits  = load_participant_features(TEXT_FEATURE_CACHE, "text", EDA_TEXT_FEATURE_CACHE)
print()
audio_splits = load_participant_features(AUDIO_FEATURE_CACHE, "audio")

## 5. Model Architectures

Identical to the flat mid-fusion notebook — `get_embedding(x)` is called once *per bucket*
instead of once per participant.

In [ ]:
# ── Text encoder (Mental-RoBERTa LSTM) ────────────────────────────────────────
class MentalRoBERTaLSTMEncoder(nn.Module):
    def __init__(self, input_dim=768, lstm_hidden=64, lstm_layers=1,
                 bidirectional=False, hidden_dim=64, dropout=0.6):
        super().__init__()
        self.lstm_out_dim = lstm_hidden * (2 if bidirectional else 1)
        self.lstm = nn.LSTM(input_dim, lstm_hidden, lstm_layers,
                            batch_first=True, bidirectional=bidirectional,
                            dropout=dropout if lstm_layers > 1 else 0.0)
        self.attention     = nn.Linear(self.lstm_out_dim, 1)
        self.norm          = nn.LayerNorm(self.lstm_out_dim)
        self.input_dropout = nn.Dropout(dropout)
        self.classifier    = nn.Sequential(
            nn.Linear(self.lstm_out_dim, hidden_dim), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim, 1),
        )

    def get_embedding(self, x: torch.Tensor) -> torch.Tensor:
        """Returns the 64-dim attention-pooled embedding (LayerNorm applied)."""
        enc, _ = self.lstm(self.input_dropout(x).unsqueeze(0))
        enc    = enc.squeeze(0)
        attn_w = torch.softmax(self.attention(enc), dim=0)
        return self.norm((attn_w * enc).sum(dim=0))   # (64,)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.get_embedding(x)).squeeze()


text_encoder = MentalRoBERTaLSTMEncoder().to(DEVICE)
print(f"Text encoder output dim : 64")
print(f"  Total params : {sum(p.numel() for p in text_encoder.parameters()):,}")

In [ ]:
# ── Audio encoder (WavLM LSTM) ────────────────────────────────────────────────
class WavLMLSTMEncoder(nn.Module):
    def __init__(self, input_dim=768, lstm_hidden=128, lstm_layers=1,
                 bidirectional=False, hidden_dim=128, dropout=0.5):
        super().__init__()
        self.lstm_out_dim = lstm_hidden * (2 if bidirectional else 1)
        self.lstm = nn.LSTM(input_dim, lstm_hidden, lstm_layers,
                            batch_first=True, bidirectional=bidirectional,
                            dropout=dropout if lstm_layers > 1 else 0.0)
        self.attention     = nn.Linear(self.lstm_out_dim, 1)
        self.norm          = nn.LayerNorm(self.lstm_out_dim)
        self.input_dropout = nn.Dropout(dropout)
        self.classifier    = nn.Sequential(
            nn.Linear(self.lstm_out_dim, hidden_dim), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(hidden_dim, 1),
        )

    def get_embedding(self, x: torch.Tensor) -> torch.Tensor:
        """Returns the 128-dim attention-pooled embedding (LayerNorm applied)."""
        enc, _ = self.lstm(self.input_dropout(x).unsqueeze(0))
        enc    = enc.squeeze(0)
        attn_w = torch.softmax(self.attention(enc), dim=0)
        return self.norm((attn_w * enc).sum(dim=0))   # (128,)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.classifier(self.get_embedding(x)).squeeze()


audio_encoder = WavLMLSTMEncoder().to(DEVICE)
print(f"Audio encoder output dim : 128")
print(f"  Total params : {sum(p.numel() for p in audio_encoder.parameters()):,}")

## 6. Load Pre-trained Weights & Freeze Encoders

In [ ]:
# Text checkpoint: raw state dict
text_encoder.load_state_dict(
    torch.load(TEXT_CKPT, map_location=DEVICE, weights_only=True)
)
# Audio checkpoint: nested dict
audio_ckpt = torch.load(AUDIO_CKPT, map_location=DEVICE, weights_only=True)
audio_encoder.load_state_dict(audio_ckpt["model_state_dict"])

for enc in (text_encoder, audio_encoder):
    enc.eval()
    for p in enc.parameters():
        p.requires_grad = False

print("Text  encoder: loaded & frozen")
print("Audio encoder: loaded & frozen")
print(f"  (audio ckpt epoch={audio_ckpt['epoch']}  "
      f"val_macro_f1={audio_ckpt['val_macro_f1']:.4f})")

## 7. Pre-compute Temporal Bucket Embeddings

**New step vs. flat mid-fusion**: instead of running each frozen encoder over *all* chunks/segments
at once per participant, we first divide the sequence into `N_BUCKETS` equal windows by relative
index, then run the encoder on each window separately.

```
For each participant, bucket b ∈ {0, …, N_BUCKETS-1}:
  t_start = b * T // N_BUCKETS        (text chunk start index)
  a_start = b * A // N_BUCKETS        (audio segment start index)
  text_emb_b  = text_encoder.get_embedding(text_feats[t_start:t_end])   → (64,)
  audio_emb_b = audio_encoder.get_embedding(audio_feats[a_start:a_end]) → (128,)
  fused_b     = concat([text_emb_b, audio_emb_b])                        → (192,)

seq = stack([fused_0, …, fused_{N-1}])   → (N_BUCKETS, 192)
```

For EDA virtual participants (train only), their augmented text chunks are paired with
the real participant's audio segments (`real_pid = virtual_pid // 1000`).

In [ ]:
TEXT_EMB_DIM  = 64
AUDIO_EMB_DIM = 128
FUSED_DIM     = TEXT_EMB_DIM + AUDIO_EMB_DIM   # 192


def extract_bucket_embeddings(split: str) -> list:
    """
    Returns list of (pid, {"seq": np.ndarray(N_BUCKETS, FUSED_DIM), "label": int})

    For each participant:
      - Divide text chunks  into N_BUCKETS windows by relative index.
      - Divide audio segs   into N_BUCKETS windows by relative index.
      - Run the frozen encoder on each window → one (64,) and (128,) per bucket.
      - Concatenate → (192,) per bucket, stack → (N_BUCKETS, 192).

    EDA virtual participants (pid > 10_000, train only):
      text window comes from the augmented chunks;
      audio window comes from the matching real participant.
    """
    text_encoder.eval()
    audio_encoder.eval()

    real_audio = {pid: data for pid, data in audio_splits[split]}
    result = []

    with torch.no_grad():
        for pid, text_data in text_splits[split]:
            # Resolve audio source
            if pid in real_audio:
                audio_data = real_audio[pid]
            elif pid > 10_000:          # EDA virtual participant
                real_pid = pid // 1000
                if real_pid not in real_audio:
                    continue
                audio_data = real_audio[real_pid]
            else:
                continue                # no matching audio

            text_feats  = np.array(text_data["feats"],  dtype=np.float32)  # (T, 768)
            audio_feats = np.array(audio_data["feats"], dtype=np.float32)  # (A, 768)
            T = len(text_feats)
            A = len(audio_feats)

            bucket_embs = []
            for b in range(N_BUCKETS):
                t_s = int(b * T / N_BUCKETS);   t_e = int((b + 1) * T / N_BUCKETS)
                a_s = int(b * A / N_BUCKETS);   a_e = int((b + 1) * A / N_BUCKETS)

                # Fall back to full sequence if a bucket would be empty
                t_chunk = text_feats[t_s:t_e]  if t_e > t_s else text_feats
                a_chunk = audio_feats[a_s:a_e] if a_e > a_s else audio_feats

                t_emb = text_encoder.get_embedding(
                    torch.tensor(t_chunk, dtype=torch.float32, device=DEVICE)
                ).cpu().numpy()   # (64,)

                a_emb = audio_encoder.get_embedding(
                    torch.tensor(a_chunk, dtype=torch.float32, device=DEVICE)
                ).cpu().numpy()   # (128,)

                bucket_embs.append(np.concatenate([t_emb, a_emb]))  # (192,)

            result.append((pid, {
                "seq":   np.stack(bucket_embs),   # (N_BUCKETS, FUSED_DIM)
                "label": text_data["label"],
            }))

    return result


print(f"Extracting temporal bucket embeddings (N_BUCKETS={N_BUCKETS}) ...")
train_data = extract_bucket_embeddings("train")
dev_data   = extract_bucket_embeddings("dev")
test_data  = extract_bucket_embeddings("test")

for name, data in [("Train", train_data), ("Dev", dev_data), ("Test", test_data)]:
    n_dep = sum(d["label"] for _, d in data)
    seq_shape = data[0][1]["seq"].shape
    print(f"{name:5s}: {len(data)} participants  dep: {n_dep}  "
          f"ctrl: {len(data)-n_dep}  seq shape: {seq_shape}")

## 8. Temporal Mid-Fusion Classifier

```
Input  : (N_BUCKETS, 192)  — temporal sequence of fused bucket embeddings
         ↓ Linear(192, 1) attention scores + softmax over N_BUCKETS
         ↓ attention-weighted sum → (192,)
         ↓ LayerNorm(192)
         ↓ Linear(192, 128) + ReLU + Dropout
         ↓ Linear(128,  64) + ReLU + Dropout
         ↓ Linear( 64,   1)
Output : scalar logit
```

The attention layer learns **which temporal window is most informative** for depression classification.
This is interpretable: inspecting the attention weights reveals which part of the interview
the model finds most diagnostic.

In [ ]:
MLP_HIDDEN = 128
DROPOUT    = 0.5


class TemporalMidFusionClassifier(nn.Module):
    """
    Applies learned attention pooling over a sequence of temporally-aligned
    fused (text + audio) bucket embeddings, then classifies with a small MLP.

    Input  : (N_BUCKETS, FUSED_DIM=192)
    Output : scalar logit
    """
    def __init__(self, fused_dim: int = FUSED_DIM,
                 hidden_dim: int = MLP_HIDDEN,
                 dropout: float = DROPOUT):
        super().__init__()
        # Scalar attention score per bucket (learned which window matters most)
        self.attention = nn.Linear(fused_dim, 1)
        self.norm      = nn.LayerNorm(fused_dim)
        self.net = nn.Sequential(
            nn.Linear(fused_dim,       hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim,      hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1),
        )

    def get_attention_weights(self, seq: torch.Tensor) -> torch.Tensor:
        """Returns softmax attention weights over the N_BUCKETS axis."""
        return torch.softmax(self.attention(seq), dim=0)   # (N_BUCKETS, 1)

    def forward(self, seq: torch.Tensor) -> torch.Tensor:
        # seq : (N_BUCKETS, FUSED_DIM)
        attn_w = self.get_attention_weights(seq)           # (N_BUCKETS, 1)
        pooled = self.norm((attn_w * seq).sum(dim=0))      # (FUSED_DIM,)
        return self.net(pooled).squeeze()


mid_model = TemporalMidFusionClassifier().to(DEVICE)
n_params  = sum(p.numel() for p in mid_model.parameters() if p.requires_grad)
print(f"TemporalMidFusionClassifier trainable parameters: {n_params:,}")

# Sanity check
with torch.no_grad():
    _dummy = torch.zeros(N_BUCKETS, FUSED_DIM, device=DEVICE)
    print(f"Output shape: {mid_model(_dummy).shape}  (expected: torch.Size([]))")

## 9. Training Configuration

In [ ]:
PARTICIPANT_BATCH_SIZE  = 16
NUM_EPOCHS              = 500
LEARNING_RATE           = 1e-4
WEIGHT_DECAY            = 1e-4
EARLY_STOPPING_PATIENCE = 50
CHECKPOINT_FREQ         = 50
GRAD_CLIP_NORM          = 1.0
AUGMENT_NOISE_STD       = 0.03   # Gaussian noise on the pre-computed bucket sequences

n_dep_train  = sum(d["label"] for _, d in train_data)
n_ctrl_train = len(train_data) - n_dep_train
pos_weight   = torch.tensor(n_ctrl_train / n_dep_train, dtype=torch.float32, device=DEVICE)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = Adam(mid_model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

print(f"Training participants : {len(train_data)}  (dep: {n_dep_train}  ctrl: {n_ctrl_train})")
print(f"pos_weight            : {pos_weight.item():.4f}")
print(f"Batch size            : {PARTICIPANT_BATCH_SIZE}")
print(f"Epochs                : {NUM_EPOCHS}  (patience {EARLY_STOPPING_PATIENCE})")
print(f"Learning rate         : {LEARNING_RATE}")
print(f"Augmentation noise    : {AUGMENT_NOISE_STD}")

## 10. Training Loop

**Augmentation**: small Gaussian noise added to the full bucket sequence `(N_BUCKETS, 192)` during
training — regularises the attention weights and the MLP.

**Early stopping**: on validation macro-F1.

**Threshold tuning**: best decision threshold tuned on the validation set after training.

In [ ]:
def train_epoch():
    mid_model.train()
    idx     = torch.randperm(len(train_data)).tolist()
    batches = [idx[i : i + PARTICIPANT_BATCH_SIZE]
               for i in range(0, len(idx), PARTICIPANT_BATCH_SIZE)]
    total_loss, correct, n = 0.0, 0, len(train_data)

    for batch in batches:
        optimizer.zero_grad()
        for i in batch:
            _, data = train_data[i]
            # seq shape: (N_BUCKETS, FUSED_DIM)
            seq   = torch.tensor(data["seq"], dtype=torch.float32, device=DEVICE)
            # Gaussian noise augmentation on the bucket sequence
            seq   = seq + torch.randn_like(seq) * AUGMENT_NOISE_STD
            label = torch.tensor(float(data["label"]), dtype=torch.float32, device=DEVICE)

            logit = mid_model(seq)
            loss  = criterion(logit, label) / len(batch)
            loss.backward()
            total_loss += loss.item() * len(batch)
            correct    += int((1 if torch.sigmoid(logit).item() >= 0.5 else 0) == data["label"])

        torch.nn.utils.clip_grad_norm_(mid_model.parameters(), GRAD_CLIP_NORM)
        optimizer.step()
    return total_loss / n, correct / n


def evaluate(data: list, threshold: float = 0.5):
    mid_model.eval()
    total_loss, records = 0.0, []
    with torch.no_grad():
        for pid, d in data:
            seq   = torch.tensor(d["seq"],   dtype=torch.float32, device=DEVICE)
            label = torch.tensor(float(d["label"]), dtype=torch.float32, device=DEVICE)
            logit = mid_model(seq)
            total_loss += criterion(logit, label).item()
            prob = torch.sigmoid(logit).item()
            records.append((pid, 1 if prob >= threshold else 0, prob, d["label"]))
    df       = pd.DataFrame(records, columns=["pid", "pred", "prob", "label"])
    acc      = accuracy_score(df["label"], df["pred"])
    macro_f1 = f1_score(df["label"], df["pred"], average="macro", zero_division=0)
    return total_loss / len(data), acc, macro_f1, df


# ── Training ─────────────────────────────────────────────────────────────────
history = {"train_loss": [], "train_acc": [],
           "val_loss":   [], "val_acc":   [], "val_f1": []}
best_val_f1       = -1.0
epochs_no_improve = 0
best_ckpt         = None

print(f"{'Epoch':>6}  {'TrainLoss':>9}  {'TrainAcc':>8}  "
      f"{'ValLoss':>7}  {'ValAcc':>6}  {'ValF1':>7}")
print("-" * 58)

for epoch in range(1, NUM_EPOCHS + 1):
    train_loss, train_acc         = train_epoch()
    val_loss, val_acc, val_f1, _  = evaluate(dev_data)

    history["train_loss"].append(train_loss)
    history["train_acc"].append(train_acc)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["val_f1"].append(val_f1)

    if epoch % CHECKPOINT_FREQ == 0:
        torch.save({"epoch": epoch, "model_state_dict": mid_model.state_dict(),
                    "val_f1": val_f1},
                   SAVE_DIR / f"epoch{epoch}_mid_fusion_temporal_a.pth")

    if val_f1 > best_val_f1:
        best_val_f1       = val_f1
        epochs_no_improve = 0
        best_ckpt = {"epoch": epoch, "val_loss": val_loss,
                     "val_acc": val_acc, "val_f1": val_f1,
                     "state": {k: v.cpu().clone()
                               for k, v in mid_model.state_dict().items()}}
        torch.save(best_ckpt["state"], SAVE_DIR / "mid_fusion_temporal_a_best.pth")
        print(f"{epoch:6d}  {train_loss:9.4f}  {train_acc:8.4f}  "
              f"{val_loss:7.4f}  {val_acc:6.4f}  {val_f1:7.4f}  *** best")
    else:
        epochs_no_improve += 1
        if epoch % 10 == 0:
            print(f"{epoch:6d}  {train_loss:9.4f}  {train_acc:8.4f}  "
                  f"{val_loss:7.4f}  {val_acc:6.4f}  {val_f1:7.4f}")

    if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}.")
        break

print(f"\nBest: epoch={best_ckpt['epoch']}  "
      f"val_loss={best_ckpt['val_loss']:.4f}  "
      f"val_f1={best_ckpt['val_f1']:.4f}")

# Restore best weights
mid_model.load_state_dict(best_ckpt["state"])
mid_model.to(DEVICE)

# Tune threshold on validation set
_, _, _, val_df = evaluate(dev_data)
best_thresh, best_f1 = 0.5, -1.0
for t in np.arange(0.10, 0.91, 0.05):
    f1 = f1_score(val_df["label"], (val_df["prob"] >= t).astype(int),
                  average="macro", zero_division=0)
    if f1 > best_f1:
        best_f1, best_thresh = f1, float(t)
print(f"Best threshold: {best_thresh:.2f}  (val macro-F1 = {best_f1:.4f})")

## 11. Training Curves

In [ ]:
epochs_ran = range(1, len(history["train_loss"]) + 1)
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].plot(epochs_ran, history["train_loss"], label="train", color="tab:blue")
axes[0].plot(epochs_ran, history["val_loss"],   label="val",   color="tab:orange")
axes[0].set_title("BCE Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()
axes[0].grid(True)

axes[1].plot(epochs_ran, history["train_acc"], label="train", color="tab:blue")
axes[1].plot(epochs_ran, history["val_acc"],   label="val",   color="tab:orange")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].set_ylim(0, 1)
axes[1].legend()
axes[1].grid(True)

axes[2].plot(epochs_ran, history["val_f1"], color="tab:green")
axes[2].set_title("Val Macro-F1")
axes[2].set_xlabel("Epoch")
axes[2].set_ylim(0, 1)
axes[2].grid(True)

plt.suptitle(f"Mid Fusion Temporal-A (N={N_BUCKETS}) — Training Curves", fontsize=13)
plt.tight_layout()
plt.show()

## 12. Test Set Evaluation

In [ ]:
_, _, _, test_df = evaluate(test_data, threshold=best_thresh)

y_true = test_df["label"].values.astype(int)
y_pred = test_df["pred"].values.astype(int)

print(f"Best checkpoint : epoch {best_ckpt['epoch']}  "
      f"val_loss={best_ckpt['val_loss']:.4f}  val_f1={best_ckpt['val_f1']:.4f}")
print(f"Decision threshold : {best_thresh:.2f}  (tuned on validation macro-F1)")
print(f"Participants : {len(test_df)}\n")
print(classification_report(y_true, y_pred,
                             target_names=["Control", "Depressed"], zero_division=0))

## 13. Confusion Matrix

In [ ]:
cm   = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(
    confusion_matrix=cm, display_labels=["Control (0)", "Depressed (1)"]
)
disp.plot(cmap="Blues")
plt.title(f"Confusion Matrix — Mid Fusion Temporal-A N={N_BUCKETS} (Test Set)")
plt.show()

## 14. Temporal Bucket Attention Weights

Inspect which temporal window the model attends to most for each test participant.
A uniform distribution across buckets suggests no strong temporal pattern;
a peaked distribution indicates the model found a specific part of the interview
most diagnostic.

In [ ]:
mid_model.eval()

# --- Per-participant attention weights on the test set ---
bucket_labels    = [f"Bucket {b}\n({b/N_BUCKETS:.0%}–{(b+1)/N_BUCKETS:.0%})" for b in range(N_BUCKETS)]
dep_attn_list    = []
ctrl_attn_list   = []

n_show = min(6, len(test_data))
fig, axes = plt.subplots(n_show, 1, figsize=(10, 2.5 * n_show))
if n_show == 1:
    axes = [axes]

with torch.no_grad():
    for ax, (pid, d) in zip(axes, test_data[:n_show]):
        seq    = torch.tensor(d["seq"], dtype=torch.float32, device=DEVICE)
        attn_w = mid_model.get_attention_weights(seq).squeeze().cpu().numpy()  # (N_BUCKETS,)
        logit  = mid_model(seq)
        prob   = torch.sigmoid(logit).item()
        pred   = 1 if prob >= best_thresh else 0

        ax.bar(range(N_BUCKETS), attn_w, color="steelblue", alpha=0.8)
        ax.set_xticks(range(N_BUCKETS))
        ax.set_xticklabels(bucket_labels, fontsize=8)
        ax.set_ylabel("Attention weight")
        status    = "CORRECT" if pred == d["label"] else "WRONG"
        label_str = "Depressed" if d["label"] == 1 else "Control"
        pred_str  = "Depressed" if pred == 1 else "Control"
        ax.set_title(
            f"Participant {pid}  |  True: {label_str}  |  "
            f"Pred: {pred_str} (p={prob:.2f})  |  {status}"
        )
        ax.grid(True, axis="y", alpha=0.4)

        if d["label"] == 1:
            dep_attn_list.append(attn_w)
        else:
            ctrl_attn_list.append(attn_w)

plt.suptitle(f"Temporal Bucket Attention Weights — Test Set (first {n_show} participants)",
             fontsize=12)
plt.tight_layout()
plt.show()

# --- Aggregate: mean attention per bucket for depressed vs. control ---
# Collect for all test participants
dep_attn_list, ctrl_attn_list = [], []
with torch.no_grad():
    for pid, d in test_data:
        seq    = torch.tensor(d["seq"], dtype=torch.float32, device=DEVICE)
        attn_w = mid_model.get_attention_weights(seq).squeeze().cpu().numpy()
        (dep_attn_list if d["label"] == 1 else ctrl_attn_list).append(attn_w)

fig, ax = plt.subplots(figsize=(8, 4))
x = np.arange(N_BUCKETS)
width = 0.35
if dep_attn_list:
    ax.bar(x - width/2, np.mean(dep_attn_list,  axis=0), width,
           label="Depressed", color="tomato",    alpha=0.8)
if ctrl_attn_list:
    ax.bar(x + width/2, np.mean(ctrl_attn_list, axis=0), width,
           label="Control",   color="steelblue", alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(bucket_labels)
ax.set_ylabel("Mean attention weight")
ax.set_title("Mean Temporal Bucket Attention — Depressed vs. Control (Test Set)")
ax.legend()
ax.grid(True, axis="y", alpha=0.4)
plt.tight_layout()
plt.show()